In [54]:
import numpy as np
from scipy.sparse import lil_matrix, csr_matrix

In [55]:
# def create_transition_matrix(num_states):
#     return np.zeros((num_states, num_states))

In [56]:
# def populate_transition_matrix(transition_matrix, transitions_data):
    

In [178]:
def state_space(U_max, S_max):
    states = []
    index_for = {}
    for u in range(U_max + 1):
        for s in range(S_max + 1):
            idx = len(states)
            states.append((u, s))
            index_for[(u, s)] = idx
            
    return states, index_for

In [179]:
def create_transition_matrix(reactions, U_max, S_max):
    states = []
    index_for = {}
    for u in range(U_max + 1):
        for s in range(S_max + 1):
            idx = len(states)
            states.append((u, s))
            index_for[(u, s)] = idx
            
    n_states = len(states)
    A = lil_matrix((n_states, n_states), dtype=float)

    for i, (u, s) in enumerate(states):
        out_rate = 0
            
        for rxn in reactions:
            u2 = u + rxn.dU
            s2 = s + rxn.dS

            if (u2 < 0) or (u2 > U_max) or (s2 < 0) or (s2 > S_max):
                continue
                    
            rate = rxn.rate_fn(u, s)
            j = index_for[(u2, s2)]
            A[i, j] += rate
            out_rate += rate
            
        A[i, i] = -out_rate
    
    return A

In [163]:
class Reaction:
    def __init__(self, dU, dS, rate_fn):
        self.dU = dU
        self.dS = dS
        self.rate_fn = rate_fn  # function: input (# u, # s) -> rate

alpha = 1.0 # Transcription rate
beta  = 0.5 # Splicing per unspliced
gamma = 0.05 # Decay per spliced

rxns = [
    # Transcription: (u, s) -> (u+1, s)
    Reaction(dU=1, dS=0, rate_fn=lambda u, s: alpha),
    
    # Splicing: (u, s) -> (u-1, s+1)
    Reaction(dU=-1, dS=1, rate_fn=lambda u, s: beta * u),
    
    # Degradation: (u, s) -> (u, s-1)
    Reaction(dU=0, dS=-1, rate_fn=lambda u, s: gamma * s),
]

In [181]:
U_max = 20
S_max = 20

A = create_transition_matrix(rxns, U_max, S_max)

In [182]:
states, index_for = state_space(U_max, S_max)

In [184]:
start_idx = 10

x0 = np.zeros(shape=(A.shape[0], 1), dtype="float")
x0[start_idx] = 1.0

In [191]:
from scipy.sparse.linalg import expm_multiply

t = 10  # time step
x_t = expm_multiply((A.T) * t, x0).T 

In [ ]:
x0 = expm_multiply((A.T) * -t, x_t).T 